In [1]:
import gymnasium as gym
from generated_kx_env import GeneratedKXEnv # Your environment class
from heuristic_agent import HeuristicAgent # The agent class

def run_simulation(env_name: str = "GeneratedKXEnv-v0", num_episodes: int = 5, max_steps_per_episode: int = 200):
    """
    Runs a simulation with the heuristic agent in the specified environment.
    """
    # Register the environment if it's not already
    try:
        gym.make(env_name)
    except gym.error.UnregisteredEnv:
        gym.envs.registration.register(
            id=env_name,
            entry_point='generated_kx_env:GeneratedKXEnv',
            max_episode_steps=max_steps_per_episode,
        )

    env = gym.make(env_name, render_mode="human")
    agent = HeuristicAgent(env) # Pass the wrapped env to the agent

    print(f"Running {num_episodes} episodes with HeuristicAgent...")

    for episode in range(num_episodes):
        observation, info = env.reset()
        total_reward = 0
        terminated = False
        truncated = False
        step_count = 0

        # Print initial state and constants after reset
        print(f"\n--- Episode {episode + 1} ---")
        # Access attributes of the unwrapped environment
        print(f"Initial State: {env.unwrapped.state}")
        print(f"Constants: {env.unwrapped.constants}")
        if env.unwrapped.last_failing_precondition_formula:
            print(f"Warning: Initial state did not satisfy precondition. Failing conjunct: {env.unwrapped.last_failing_precondition_formula}")

        while not terminated and not truncated and step_count < max_steps_per_episode:
            action = agent.choose_action(observation)
            
            # Print chosen action for debugging
            # print(f"Step {step_count}: Chosen Action: {action}")

            observation, reward, terminated, truncated, info = env.step(action)
            total_reward += reward
            step_count += 1

            env.render() # Render the environment (prints state to console)

            # Print current state and reward
            # print(f"Step {step_count}: Reward={reward:.2f}, Total Reward={total_reward:.2f}, State={env.unwrapped.state}")

        print(f"Episode {episode + 1} finished after {step_count} steps.")
        # Access attributes of the unwrapped environment
        print(f"Final State: {env.unwrapped.state}")
        print(f"Total Reward: {total_reward:.2f}")
        print(f"Terminated: {terminated}, Truncated: {truncated}")
        if terminated and total_reward > 0: # Check total_reward to differentiate success from safety termination
            print("Episode SUCCESS: Agent reached the goal (avoided obstacle successfully)!")
        elif terminated:
            print("Episode FAILURE: Agent terminated due to safety violation or other failure.")
        elif truncated:
            print("Episode TRUNCATED: Agent ran out of steps.")

    env.close()
    print("\nSimulation complete.")


In [2]:
print("Starting simulation with HeuristicAgent in GeneratedKXEnv...")
run_simulation(num_episodes=3)

Starting simulation with HeuristicAgent in GeneratedKXEnv...
Initial state set by Z3 (possibly perturbed): {'x': -1.0, 'y': 0.0, 'v': 0.0, 'a': 0.0, 'dx': -1.0, 'dy': 0.0, 'w': 0.0, 'r': 1.0, 'xo': 0.0, 'yo': 0.0, 't': 0.0}
Constants possibly set by Z3: {'ep': 1.0, 'b': 1.0, 'A': 0.0, 'W': 0.0}
Running 3 episodes with HeuristicAgent...
Initial state set by Z3 (possibly perturbed): {'x': -1.0, 'y': 0.0, 'v': 0.0, 'a': 0.0, 'dx': -1.0, 'dy': 0.0, 'w': 0.0, 'r': 1.0, 'xo': 0.0, 'yo': 0.0, 't': 0.0}
Constants possibly set by Z3: {'ep': 1.0, 'b': 1.0, 'A': 0.0, 'W': 0.0}

--- Episode 1 ---
Initial State: {'x': -1.0, 'y': 0.0, 'v': 0.0, 'a': 0.0, 'dx': -1.0, 'dy': 0.0, 'w': 0.0, 'r': 1.0, 'xo': 0.0, 'yo': 0.0, 't': 0.0}
Constants: {'ep': 1.0, 'b': 1.0, 'A': 0.0, 'W': 0.0}
Error evaluating expression '1/v if v > 0 else v' (transformed: '(1/self.state['v'] if self.state['v']) > (0 else self.state['v'])'): expected 'else' after 'if' expression (<string>, line 1)
Render: Step=1, State={'x': np.f